# Advanced GPU Project 2 — Fine-tune DistilBERT for Emotion Classification
## NLP • Hugging Face Transformers • GPU recommended

**Task:** Predict emotion labels from short text using a pretrained language model.  
**Dataset:** `dair-ai/emotion` from Hugging Face.  
**Runtime:** Google Colab GPU recommended.

This is a practical transformer fine-tuning workflow: tokenization, batching, validation, macro-F1, error analysis, and inference.

## 1. Verify GPU

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Enable GPU under Runtime → Change runtime type in Colab.")

In [ ]:
%pip -q install -U transformers datasets accelerate evaluate scikit-learn

## 2. Load the dataset

In [ ]:
from datasets import load_dataset
from collections import Counter

dataset = load_dataset("dair-ai/emotion")
label_names = dataset["train"].features["label"].names
print(dataset)
print("Labels:", label_names)
print("Train distribution:", Counter(dataset["train"]["label"]))
display(dataset["train"][:5])

## 3. Tokenize text

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized = tokenized.remove_columns(["text"])
print(tokenized)

## 4. Load pretrained model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=len(label_names),
    id2label={i: n for i, n in enumerate(label_names)},
    label2id={n: i for i, n in enumerate(label_names)}
)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 5. Configure metrics and Trainer

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, pred),
        "macro_f1": f1_score(labels, pred, average="macro")
    }

args = TrainingArguments(
    output_dir="distilbert-emotion-checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

## 6. Fine-tune

In [ ]:
trainer.train()

## 7. Evaluate on the test set

In [ ]:
result = trainer.predict(tokenized["test"])
y_true = result.label_ids
y_pred = np.argmax(result.predictions, axis=-1)
print("Test metrics:", result.metrics)

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=label_names, xticks_rotation=45, cmap="Blues"
)
plt.tight_layout()
plt.show()

## 8. Predict custom sentences

In [ ]:
texts = [
    "I am thrilled that our team reached the final round!",
    "I am scared that the system will fail during the demo.",
    "I am angry about the unfair result.",
    "I feel sad that the project did not work."
]
model = trainer.model
model.eval()
batch = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
batch = {k: v.to(model.device) for k, v in batch.items()}
with torch.no_grad():
    logits = model(**batch).logits
probs = torch.softmax(logits, dim=-1)
for text, p in zip(texts, probs):
    idx = int(p.argmax())
    print(f"{text}\n→ {label_names[idx]} ({p[idx].item():.1%})\n")

## 9. Save model and tokenizer

In [ ]:
trainer.save_model("distilbert-emotion-model")
tokenizer.save_pretrained("distilbert-emotion-model")
print("Saved distilbert-emotion-model/")

## Challenge extensions
- Compare DistilBERT against TF-IDF + Logistic Regression.
- Tune sequence length, learning rate, or number of epochs.
- Examine confusing label pairs and per-class F1.
- Build a small Gradio text interface.
- Discuss dataset bias and why predicted emotion is not a definitive psychological assessment.

**Submit:** notebook, test macro-F1, confusion matrix, error analysis, model artifact, and demo.